# PCP focused evaluation — A100 worker 0

The run cell first requires the separately logged 400-episode Jeff stock replication to match every historical outcome. Recorded matmul settings match Jeff: matmul TF32 off, highest precision; cuDNN TF32 retains the Torch default because Jeff did not record it; initial seed stream 0; batches grouped by task and historical shard parity. Existing results remain under their old experiment. Open worker 0 and worker 1 in **separate A100 Colab runtimes**. Add `SUPABASE_URL` and `SUPABASE_SERVICE_KEY` to Colab Secrets, enable notebook access, then **Run all**. The code bundle and three frozen checkpoints download automatically from Supabase; there is no file-upload prompt. `HF_TOKEN` is optional for public assets. No GitHub checkout or manual file upload is needed. The notebook pins the bundle and each checkpoint by SHA256, verifies them before loading, and caches the download for reruns in this runtime.

CNN MC is excluded based on weaker previous gradient-direction evidence. These two critics are not established as universally best. The fixed grid has **18 configurations, 9 per worker**, plus each worker's paired **stock SmolVLA baseline: no P&P, no re-noising, no Q correction**. Treatments retain their P&P parent (steps 1/2/3, K 3/1/1) plus Q correction; this measures the whole treatment against stock, not Q's marginal benefit alone. All configurations apply PCP at sampler step 3, every action chunk throughout the full episode. Two modest standardized RMS radii (.02/.06): ascent, descent, and Q<0.5 gated ascent for frozen MC and continued TD (12 arms); matched random controls (2 arms); three-inner-step ascent for frozen MC and TD (4 arms). Most settings are fixed to prioritize depth over breadth. 

No wall-clock budget or cutoff, and no adaptive elimination. Every arm receives the same cases: up to 50 released starting states on each of 40 tasks, with three shared generation-noise streams (0/1/2; stream 0 matches Jeff). Normally **6,000 episodes per configuration**, or **60,000 full-episode rollouts per worker** including its zero baseline. Progress milestones are 400 / 2,000 / 6,000 cases; if the benchmark exposes fewer states, the plan adjusts automatically. ETA is measured from throughput, not guaranteed to fit 12 hours. Rerunning the same worker resumes completed Supabase rows.

Leaderboards include successes, rescues/spoils, conditional rescue/spoil rates, paired success change and standard errors clustered by task/starting state across repeated seeds. They also count actual PCP interventions. Chunks and repeated seeds are not counted as independent starting states. Both workers use the same cohort and shared baseline seeds; their baselines must not be pooled as independent replicates.

Batching starts at eight, grouped by task and historical shard parity, with parallel simulator processes, reused environments, matmul TF32 disabled, sparse camera rendering, and four background upload clients. Torch is pinned to 2.11.0+cu128. This does not reproduce every historical resume boundary; the dedicated replication runner reconstructs those separately. No critic classification/Bellman-validation phase, smoke rollout or video encoding. Actor CPU work and rendering still limit utilization; reported GPU utilization distinguishes that from inference limits.

Completed outcomes go to Supabase `rollouts` under `smolvla-pcp-a100-jeff-settings-stock-v6`, with configuration/checkpoint identity, errors, success and steps. Storage `artifacts/smolvla_pcp_sweep/...` contains the manifest, per-chunk correction telemetry, executed trajectories and worker leaderboards. Use the companion monitor notebook from any CPU Colab session while these workers run.

This is fixed, equal-exposure exploratory evaluation on released benchmark states, some of which may have been used previously. Many interventions within an episode do not create independent evidence. Standard errors are conditional on these 40 benchmark tasks and group seed repeats by starting state; they do not establish generalization to new tasks or remove multiple-comparison bias. Correction radii use the frozen MC training action standard deviations as a common coordinate system for every critic and random control. CNN's anchor is the live probe clean estimate, held fixed during its inner search.


In [ ]:
WORKER_INDEX = 0
WORKER_COUNT = 2
BATCH_SIZE = 8  # Jeff-matched replication setting; do not enlarge before comparison
SIMULATOR_PROCESSES = min(8, max(1, (__import__('os').cpu_count() or 2) - 2))
EXPERIMENT = 'smolvla-pcp-a100-jeff-settings-stock-v6'
BUNDLE_PATH = ''  # empty automatically downloads the exact pinned Supabase bundle
BUNDLE_BUCKET = 'artifacts'
BUNDLE_SHA256 = '671fe1c691fa072589462378c857e03c808f667c7e48f1fa189351e0252d37d4'
BUNDLE_STORAGE_KEY = 'smolvla_pcp_sweep/bundles/671fe1c691fa072589462378c857e03c808f667c7e48f1fa189351e0252d37d4/trial_bundle.zip'
CHECKPOINT_SHA256 = {'frozen_mc': '9b8800bc9f27116afcb1812f93f8a9f6767ac012c7cdac790502032fbb7d0c8e', 'cnn_mc': 'afcf832d43ffca44af843e4e5301fa0a84b1b2e485703e2a1dc13aa02b260151', 'td': '1e3fa3f093a711d0651b69e7f152a5b56a7e7a85ec754c45b0afe5834d9f324e'}
OUTPUT_DIR = f'/content/pcp-search-worker-{WORKER_INDEX}'


In [ ]:
import os, sys, subprocess, zipfile
from pathlib import Path
os.environ.update(MUJOCO_GL='egl', NVIDIA_DRIVER_CAPABILITIES='compute,utility,graphics',
                  TOKENIZERS_PARALLELISM='false', OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1')
from google.colab import userdata
for key in ('SUPABASE_URL','SUPABASE_SERVICE_KEY','HF_TOKEN'):
    try:
        value=userdata.get(key)
        if value: os.environ[key]=value
    except Exception: pass
for key in ('SUPABASE_URL','SUPABASE_SERVICE_KEY'):
    if not os.environ.get(key): raise RuntimeError(f'Add {key} in Colab Secrets and allow notebook access.')
if not BUNDLE_PATH:
    subprocess.run([sys.executable,'-m','pip','install','-q','supabase'],check=True)
    from supabase import create_client
    import hashlib
    client=create_client(os.environ['SUPABASE_URL'],os.environ['SUPABASE_SERVICE_KEY'])
    cached=Path('/content')/f'pcp_bundle_{BUNDLE_SHA256}.zip'
    if not cached.exists() or hashlib.sha256(cached.read_bytes()).hexdigest()!=BUNDLE_SHA256:
        data=client.storage.from_(BUNDLE_BUCKET).download(BUNDLE_STORAGE_KEY)
        if hashlib.sha256(data).hexdigest()!=BUNDLE_SHA256:
            raise RuntimeError('Downloaded bundle SHA256 mismatch')
        cached.write_bytes(data)
    BUNDLE_PATH=str(cached)
if __import__('hashlib').sha256(Path(BUNDLE_PATH).read_bytes()).hexdigest()!=BUNDLE_SHA256:
    raise RuntimeError('Bundle differs from the exact version pinned in this notebook')
PACKAGE=Path('/content/smolvla-pcp-trial')
PACKAGE.mkdir(exist_ok=True)
with zipfile.ZipFile(BUNDLE_PATH) as archive:
    if any(Path(name).is_absolute() or '..' in Path(name).parts for name in archive.namelist()):
        raise ValueError('Unsafe bundle path')
    archive.extractall(PACKAGE)
print({'python':sys.version,'executable':sys.executable},flush=True)
subprocess.run([sys.executable,'-m','pip','--version'],check=True)
install=subprocess.run([sys.executable,'-m','pip','install','-e',str(PACKAGE)+'[sim]',
                        'torch==2.11.0+cu128','torchvision==0.26.0+cu128',
                        '--extra-index-url','https://download.pytorch.org/whl/cu128'],
                       stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
install_log=Path('/content/pcp_dependency_install.log')
install_log.write_text(install.stdout)
if install.returncode:
    print('\n'.join(install.stdout.splitlines()[-120:]),flush=True)
    raise RuntimeError(f'Dependency installation failed (exit {install.returncode}). '
                       f'Full log: {install_log}. Paste the ERROR lines above; no experiment has started.')
print('Simulation/model dependencies installed.',flush=True)
subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=False,stdout=subprocess.DEVNULL)
if str(PACKAGE) not in sys.path: sys.path.insert(0,str(PACKAGE))
subprocess.run([sys.executable,'-m','pip','install',
                'torch==2.11.0+cu128','torchvision==0.26.0+cu128',
                '--index-url','https://download.pytorch.org/whl/cu128'],check=True)
if 'torch' in sys.modules and sys.modules['torch'].__version__ != '2.11.0+cu128':
    raise RuntimeError('Torch was already imported. Restart the runtime and rerun with the pinned version.')
from pnp import env_setup
if env_setup._find_nvidia_egl_library() is None:
    import re
    driver=subprocess.check_output(['nvidia-smi','--query-gpu=driver_version','--format=csv,noheader'],text=True).splitlines()[0]
    major=re.match(r'\d+',driver.strip()).group()
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','--no-install-recommends',f'libnvidia-gl-{major}'],check=True)
env_setup._ensure_nvidia_egl_vendor()
if env_setup._find_nvidia_egl_library() is None: raise RuntimeError('NVIDIA EGL missing; hardware rendering is required.')
env_setup._ensure_libero_config()
env_setup._fix_torch_quant_compat()
env_setup._remove_broken_optional_torchaudio()
env_setup._ensure_libero_assets(token=os.getenv('HF_TOKEN'))
import torch, hashlib, json
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime.')
metadata=json.loads((PACKAGE/'bundle_manifest.json').read_text())
if metadata['checkpoints']!=CHECKPOINT_SHA256: raise RuntimeError('Unexpected checkpoint manifest')
CHECKPOINT_PATHS={name:str(PACKAGE/'checkpoints'/f'{name}.pt') for name in metadata['checkpoints']}
for name,path in CHECKPOINT_PATHS.items():
    if hashlib.sha256(Path(path).read_bytes()).hexdigest()!=metadata['checkpoints'][name]:
        raise RuntimeError(f'Checkpoint mismatch: {name}')
print({'gpu':torch.cuda.get_device_name(0),'batch_size':BATCH_SIZE,'worker':WORKER_INDEX,'simulator_processes':SIMULATOR_PROCESSES})


In [ ]:
from pnp.smolvla_jeff_replication import require_replication
print(require_replication())
from pnp.smolvla_pcp_sweep import run_sweep, sweep_grid
# Retain shared random controls; divide the surviving 18 arms evenly.
GRID = [arm for arm in sweep_grid() if arm['critic'] in ('frozen_mc', 'td', 'none')]
for ordinal, arm in enumerate(GRID): arm['ordinal'] = ordinal
report=run_sweep(checkpoint_paths=CHECKPOINT_PATHS, worker_index=WORKER_INDEX,
                 worker_count=WORKER_COUNT, batch_size=BATCH_SIZE,
                 simulator_processes=SIMULATOR_PROCESSES,
                 output_dir=OUTPUT_DIR, experiment=EXPERIMENT, grid_override=GRID)
